# Approximate Inference in Bayesian Networks

We have shown how random sampling can be used to perform simple estimate tasks. Let us now see how to apply this to a Bayesian Network.

For ease, we will demonstrate this on the following simple Bayesian Network

[Simple Bayesian Network](../images/dag.png)

(This networks represents the relationship between studying **G**eography or **M**athematics and careers in **F**inance or **P**rogramming). The distributions are:

$P(G) = \begin{pmatrix}0.6\\0.4\end{pmatrix}$

$P(M) = \begin{pmatrix}0.75\\0.25\end{pmatrix}$

$P(F\vert M) = \begin{pmatrix}0.8 & 0.4\\ 0.2 & 0.6\end{pmatrix}$

$P(P\vert M) = \begin{pmatrix}0.65 & 0.45\\ 0.35 & 0.55\end{pmatrix}$

We could quite easily use direct computation to compute any distribution we could possibly want here, but we can also use *sampling* to compute. Indeed, when a problem becomes very large (many variable, many possibilities for each variable) then we will have to use sampling.

As we did with the simple example of the coin toss, we will draw samples and count outcomes to build up an approximation of the distribution. Here, this is to approxmate the joint distribution. However, we only have the factorisation to work with. How do we do this?

We can sequentially sample from the terms in the factorised distribution. Starting with the independent variable, we draw a sample. We then use this draw to condition subsequent distributions until we have taken a draw from all of the factors. Let's work through an example:

* Draw one sample from $P(G) = (0.6,0.4)$. Answer is (say) False.
* Draw one sample from $P(M) = (0.75,0.25)$. Answer is (say) True.
* Draw one sample from $P(F\vert M=True) = (0.4,0.6)$. Answer is True.
* Draw one sample from $P(P\vert M=True) = (0.45,0.55)$. Answer is False.

We have now drawn one sample from the joint distribution and obtained (False,True,True,False). Given a large number of such samples, we can compute anything we want. Let's try it out. We will code this up very explicitly which will be slightly inefficient but we want to maintain transparency whilst we are learning.

In [19]:
import numpy as np

PG = np.array([0.6,0.4])
PM = np.array([0.75,0.25])
PF_M = np.array([[0.8,0.4],[0.2,0.6]])
PP_M = np.array([[0.65,0.45],[0.35,0.55]])

In [20]:
N = 50000
Trace = np.zeros([N,4])
for i in range(N):
    # Draw one sample from P(G)
    G = np.random.binomial(1,PG[1])
    # Draw one sample from P(M)
    M = np.random.binomial(1,PM[1])
    # Draw one sample from P(F|M)
    F = np.random.binomial(1,PF_M[1,M])
    # Draw one sample from P(P|M)
    P = np.random.binomial(1,PP_M[1,M])
    Trace[i:] = np.array([G,M,F,P])

print(Trace)

[[1. 0. 1. 0.]
 [1. 0. 0. 0.]
 [1. 1. 0. 1.]
 ...
 [0. 0. 1. 0.]
 [0. 1. 0. 1.]
 [0. 0. 1. 0.]]


Let's do some basic stats on the Trace

In [21]:
PF = PF_M@PM
PP = PP_M@PM

print(f"P_G = {Trace[:,0].sum()/N} (expected {PG[1]})")
print(f"P_M = {Trace[:,1].sum()/N} (expected {PM[1]})")
print(f"P_F = {Trace[:,2].sum()/N} (expected {PF[1]})")
print(f"P_P = {Trace[:,3].sum()/N} (expected {PP[1]})")

P_G = 0.40224 (expected 0.4)
P_M = 0.2512 (expected 0.25)
P_F = 0.30108 (expected 0.30000000000000004)
P_P = 0.40412 (expected 0.39999999999999997)


Pretty good! Can we now estimate some other quantities? For example, how about estimating $P(M\vert F)$?

First of all, we can do this exactly:

In [22]:
PM_F = (PF_M*PM).T/PF
print(PM_F)

[[0.85714286 0.5       ]
 [0.14285714 0.5       ]]


How do we get this from the trace? We need to
* Find the rows in the Trace corresponding to the desired condition
* Then compute the proportion of those rows with the desired outcome 

In [23]:
# Find the rows where F is True
FTrue = Trace[np.where(Trace[:,2]==1)]
# Add up column 1 and divide by the number of samples where F is True
MTrueFTrue = FTrue[:,1].sum()/FTrue.shape[0]
print(f"P(M=True|F=True) = {MTrueFTrue} ({PM_F[1][1]} expected)")

FFalse = Trace[np.where(Trace[:,2]==0)]
# Add up column 1 and divide by the number of samples where F is True
MTrueFFalse = FFalse[:,1].sum()/FFalse.shape[0]
print(f"P(M=True|F=False) = {MTrueFFalse} ({PM_F[1][0]} expected)")


P(M=True|F=True) = 0.4980736017005447 (0.4999999999999999 expected)
P(M=True|F=False) = 0.14485205746008126 (0.14285714285714285 expected)


This is an example of **rejection sampling**. In each case we have rejected all the rows in the Trace for which desired condition on $F$ was false. We may consider this to be a waste; however, if we want to ask general question of the distribution, the full trace in which all variables are drawn allows us to do so, as we demonstrated by computing by $P(M\vert F)$ and $P(M\vert \lnot F)$

Rejection sampling is very general and it can be shown that the standard deviation in the error in each probability is proportional to $1/\sqrt{N}$. However, if we have specific questions we want to answer and are time or compute-limited, then we may wish for something more efficient. This will be our next topic.